In [16]:
# ============================================================
# 서울시 역사마스터 ↔ CELL 250m 매핑 최종 코드
# - station -> nearest cell
# - cell -> nearest station
# - 거리 요약 / outlier 확인
# ============================================================

import os
import pandas as pd
import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree

# =========================
# 0. 경로 설정
# =========================

cell_csv_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
station_csv_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울시 역사마스터 정보.csv"

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_seoul_master_final"
os.makedirs(output_dir, exist_ok=True)

station_to_cell_path = os.path.join(output_dir, "station_to_nearest_cell.csv")
cell_to_station_path = os.path.join(output_dir, "cell_to_nearest_station.csv")
station_outlier_path = os.path.join(output_dir, "station_mapping_outliers_over_2km.csv")
cell_outlier_path = os.path.join(output_dir, "cell_mapping_outliers_over_5km.csv")


# =========================
# 1. CSV 안전 읽기
# =========================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / encoding={enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


station_df = read_csv_safe(station_csv_path)
cell_df = read_csv_safe(cell_csv_path)

print("\n[STATION RAW]")
print(station_df.head())
print(station_df.columns)

print("\n[CELL RAW]")
print(cell_df.head())
print(cell_df.columns)


# =========================
# 2. 컬럼 정리
# =========================

station_df = station_df.rename(columns={
    "역사_ID": "STATION_ID",
    "역사명": "station_name",
    "호선": "line",
    "위도": "lat",
    "경도": "lon"
})

cell_df = cell_df.rename(columns={
    "cell_id": "CELL_ID",
    "Cell_ID": "CELL_ID",
    "x": "CELL_X",
    "y": "CELL_Y",
    "X": "CELL_X",
    "Y": "CELL_Y"
})

station_df = station_df[["STATION_ID", "station_name", "line", "lat", "lon"]].copy()
cell_df = cell_df[["CELL_ID", "CELL_X", "CELL_Y"]].copy()

station_df["lat"] = pd.to_numeric(station_df["lat"], errors="coerce")
station_df["lon"] = pd.to_numeric(station_df["lon"], errors="coerce")
cell_df["CELL_X"] = pd.to_numeric(cell_df["CELL_X"], errors="coerce")
cell_df["CELL_Y"] = pd.to_numeric(cell_df["CELL_Y"], errors="coerce")

station_df = station_df.dropna(subset=["lat", "lon"])
cell_df = cell_df.dropna(subset=["CELL_X", "CELL_Y"])

station_df = station_df.drop_duplicates(subset=["STATION_ID", "station_name", "line", "lat", "lon"])
cell_df = cell_df.drop_duplicates(subset=["CELL_ID"])

print("\n[STATION CLEAN]")
print(station_df.shape)
print(station_df.head())

print("\n[CELL CLEAN]")
print(cell_df.shape)
print(cell_df.head())


# =========================
# 3. 역 좌표 WGS84 → EPSG:5179
# =========================

transformer = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)

station_df["station_x_5179"], station_df["station_y_5179"] = transformer.transform(
    station_df["lon"].values,
    station_df["lat"].values
)

print("\n[STATION COORD CHECK]")
print(station_df[["station_x_5179", "station_y_5179"]].describe())


# =========================
# 4. KDTree 준비
# =========================

cell_points = cell_df[["CELL_X", "CELL_Y"]].to_numpy()
station_points = station_df[["station_x_5179", "station_y_5179"]].to_numpy()

cell_tree = cKDTree(cell_points)
station_tree = cKDTree(station_points)


# ============================================================
# 5. station -> nearest cell 매핑
# ============================================================

station_dist, station_idx = cell_tree.query(station_points, k=1)

matched_cells = cell_df.iloc[station_idx].reset_index(drop=True)

station_to_cell = station_df.reset_index(drop=True).copy()
station_to_cell["matched_CELL_ID"] = matched_cells["CELL_ID"].values
station_to_cell["matched_CELL_X"] = matched_cells["CELL_X"].values
station_to_cell["matched_CELL_Y"] = matched_cells["CELL_Y"].values
station_to_cell["distance_m"] = station_dist

station_to_cell["valid_within_500m"] = station_to_cell["distance_m"] <= 500
station_to_cell["valid_within_1km"] = station_to_cell["distance_m"] <= 1000
station_to_cell["valid_within_2km"] = station_to_cell["distance_m"] <= 2000

station_to_cell.to_csv(station_to_cell_path, index=False, encoding="utf-8-sig")

station_outliers = station_to_cell[station_to_cell["distance_m"] > 2000].copy()
station_outliers.to_csv(station_outlier_path, index=False, encoding="utf-8-sig")


# ============================================================
# 6. cell -> nearest station 매핑
# ============================================================

cell_dist, cell_idx = station_tree.query(cell_points, k=1)

nearest_stations = station_df.iloc[cell_idx].reset_index(drop=True)

cell_to_station = cell_df.reset_index(drop=True).copy()
cell_to_station["STATION_ID"] = nearest_stations["STATION_ID"].values
cell_to_station["nearest_station"] = nearest_stations["station_name"].values
cell_to_station["nearest_line"] = nearest_stations["line"].values
cell_to_station["station_lat"] = nearest_stations["lat"].values
cell_to_station["station_lon"] = nearest_stations["lon"].values
cell_to_station["station_x_5179"] = nearest_stations["station_x_5179"].values
cell_to_station["station_y_5179"] = nearest_stations["station_y_5179"].values
cell_to_station["nearest_station_distance_m"] = cell_dist

cell_to_station["station_access_250m"] = cell_to_station["nearest_station_distance_m"] <= 250
cell_to_station["station_access_500m"] = cell_to_station["nearest_station_distance_m"] <= 500
cell_to_station["station_access_1km"] = cell_to_station["nearest_station_distance_m"] <= 1000
cell_to_station["station_access_2km"] = cell_to_station["nearest_station_distance_m"] <= 2000
cell_to_station["station_access_5km"] = cell_to_station["nearest_station_distance_m"] <= 5000

cell_to_station["station_accessibility_score"] = 1 / (
    cell_to_station["nearest_station_distance_m"] + 1
)

cell_to_station["is_station_area_5km"] = cell_to_station["nearest_station_distance_m"] <= 5000

cell_to_station["valid_nearest_station"] = cell_to_station["nearest_station"]
cell_to_station.loc[
    ~cell_to_station["is_station_area_5km"],
    "valid_nearest_station"
] = np.nan

cell_to_station.to_csv(cell_to_station_path, index=False, encoding="utf-8-sig")

cell_outliers = cell_to_station[cell_to_station["nearest_station_distance_m"] > 5000].copy()
cell_outliers.to_csv(cell_outlier_path, index=False, encoding="utf-8-sig")


# ============================================================
# 7. 결과 확인
# ============================================================

print("\n============================================================")
print("DONE")
print("============================================================")

print("\n저장 파일")
print("station -> cell:", station_to_cell_path)
print("cell -> station:", cell_to_station_path)
print("station outliers:", station_outlier_path)
print("cell outliers:", cell_outlier_path)

print("\n[STATION -> CELL HEAD]")
print(station_to_cell.head())

print("\n[STATION -> CELL DISTANCE]")
print(station_to_cell["distance_m"].describe())

print("\n[STATION -> CELL VALID RATIO]")
print(station_to_cell[[
    "valid_within_500m",
    "valid_within_1km",
    "valid_within_2km"
]].mean())

print("\n[STATION OUTLIERS > 2km]")
print(station_outliers[[
    "STATION_ID",
    "station_name",
    "line",
    "lat",
    "lon",
    "matched_CELL_ID",
    "distance_m"
]].sort_values("distance_m", ascending=False).head(30))

print("\n[CELL -> STATION HEAD]")
print(cell_to_station.head())

print("\n[CELL -> STATION DISTANCE]")
print(cell_to_station["nearest_station_distance_m"].describe())

print("\n[CELL ACCESS RATIO]")
print(cell_to_station[[
    "station_access_250m",
    "station_access_500m",
    "station_access_1km",
    "station_access_2km",
    "station_access_5km",
    "is_station_area_5km"
]].mean())

print("\n[CELL OUTLIERS > 5km]")
print(cell_outliers[[
    "CELL_ID",
    "CELL_X",
    "CELL_Y",
    "nearest_station",
    "nearest_line",
    "nearest_station_distance_m"
]].sort_values("nearest_station_distance_m", ascending=False).head(30))

print("\n[UNIQUE STATION COUNT]")
print("station rows:", len(station_to_cell))
print("unique STATION_ID:", station_to_cell["STATION_ID"].nunique())
print("unique station names:", station_to_cell["station_name"].nunique())

print("\n[TOP MAPPED STATIONS BY CELL COUNT]")
station_cell_count = (
    cell_to_station
    .groupby(["STATION_ID", "nearest_station", "nearest_line"], as_index=False)
    .agg(
        cell_count=("CELL_ID", "count"),
        mean_distance=("nearest_station_distance_m", "mean"),
        min_distance=("nearest_station_distance_m", "min"),
        max_distance=("nearest_station_distance_m", "max"),
        within_1km=("station_access_1km", "sum"),
        within_5km=("station_access_5km", "sum")
    )
    .sort_values("cell_count", ascending=False)
)

print(station_cell_count.head(30))

station_cell_count.to_csv(
    os.path.join(output_dir, "station_cell_count_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

read success: 서울시 역사마스터 정보.csv / encoding=cp949
read success: CELL_250x250(UTMK_epsg 5179).csv / encoding=utf-8-sig

[STATION RAW]
   역사_ID 역사명          호선        위도         경도
0   9010  동탄  수도권 광역급행철도  37.20034  127.09569
1   9009  구성  수도권 광역급행철도  37.29913  127.10389
2   9008  성남  수도권 광역급행철도  37.39467  127.12058
3   9007  수서  수도권 광역급행철도  37.48637  127.10161
4   9006  삼성  수도권 광역급행철도  37.50887  127.06324
Index(['역사_ID', '역사명', '호선', '위도', '경도'], dtype='object')

[CELL RAW]
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125
Index(['CELL_ID', 'CELL_X', 'CELL_Y'], dtype='object')

[STATION CLEAN]
(783, 5)
   STATION_ID station_name        line       lat        lon
0        9010           동탄  수도권 광역급행철도  37.20034  127.09569
1        9009           구성  수도권 광역급행철도  37.29913  127.10389
2        9008           성남  수도권 광역급행철도  37.39467  127.12058
3        9007   

In [1]:
# ============================================================
# HIGH-CONFIDENCE STATION MATCH PIPELINE
# 서울교통공사_역명 지하철역 검색.csv = 매칭용 dictionary
# 매칭되는 STATION_ID만 사용
# unmatched는 버림
# ============================================================

import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로
# =========================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\research_high_confidence"
os.makedirs(output_dir, exist_ok=True)

# 매칭용 dictionary
dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

# 최종 cell-station mapping
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"

# mobility 1개 테스트
mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

# station activity 데이터
station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"

station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

# 저장 파일
dict_clean_path = os.path.join(output_dir, "station_dictionary_high_confidence.csv")
mapping_hc_path = os.path.join(output_dir, "cell_station_mapping_high_confidence.csv")
final_path = os.path.join(output_dir, "TEST_high_confidence_final_dataset.csv")
top_hidden_path = os.path.join(output_dir, "TEST_high_confidence_top_hidden.csv")
station_summary_path = os.path.join(output_dir, "TEST_high_confidence_station_summary.csv")


# =========================
# 1. 유틸
# =========================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


def clean_station_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.strip()
        .str.replace(" ", "", regex=False)
    )


# =========================
# 2. dictionary 로드
# =========================

station_dict = read_csv_safe(dict_path)

print("\n[DICT RAW]")
print(station_dict.shape)
print(station_dict.columns.tolist())
print(station_dict.head())

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_station_name(station_dict["station_name"])
station_dict["external_code_raw"] = station_dict["external_code"].astype(str).str.strip()

# 외부코드가 숫자인 것만 실제 station activity ID 후보로 사용
station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code_raw"],
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code_raw"
]].drop_duplicates()

station_dict.to_csv(dict_clean_path, index=False, encoding="utf-8-sig")

print("\n[DICT CLEAN]")
print(station_dict.shape)
print(station_dict.head())
print("saved:", dict_clean_path)


# =========================
# 3. station activity 로드
# =========================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

station_30["STATION_ID"] = pd.to_numeric(station_30["STATION_ID"], errors="coerce").astype("Int64")
station_daily["STATION_ID"] = pd.to_numeric(station_daily["STATION_ID"], errors="coerce").astype("Int64")

station_30_ids = set(station_30["STATION_ID"].dropna().astype(int).unique())
station_daily_ids = set(station_daily["STATION_ID"].dropna().astype(int).unique())
activity_ids = station_30_ids | station_daily_ids

print("\n[ACTIVITY ID CHECK]")
print("30min station ids:", len(station_30_ids))
print("daily station ids:", len(station_daily_ids))
print("union activity ids:", len(activity_ids))

# dictionary 중 activity에 실제 존재하는 ID만 retained
station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[DICT VALID IN ACTIVITY]")
print(station_dict_valid.shape)
print(station_dict_valid.head())


# =========================
# 4. mapping 로드 + dictionary로 high-confidence 매칭
# =========================

mapping = read_csv_safe(mapping_path)

print("\n[MAPPING RAW]")
print(mapping.shape)
print(mapping.columns.tolist())
print(mapping.head())

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

mapping["nearest_station_clean"] = clean_station_name(mapping["nearest_station"])

# 역명 기준 dictionary 매칭
# 매칭 안 되면 버림
mapping_hc = mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

# 중복 대응 제거:
# 같은 CELL_ID가 여러 역 ID로 붙으면 nearest_line과 dict_line 정보가 애매할 수 있으므로
# 일단 동일 cell 중 첫 번째만 사용. 필요하면 나중에 line rule 추가.
mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

mapping_hc.to_csv(mapping_hc_path, index=False, encoding="utf-8-sig")

print("\n[HIGH CONFIDENCE MAPPING]")
print("original valid mapping rows:", len(mapping))
print("high-confidence rows:", len(mapping_hc))
print("kept ratio:", len(mapping_hc) / len(mapping))
print("unique STATS_STATION_ID:", mapping_hc["STATS_STATION_ID"].nunique())
print(mapping_hc[[
    "CELL_ID",
    "nearest_station",
    "nearest_line",
    "MASTER_STATION_ID",
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "nearest_station_distance_m"
]].head(30))
print("saved:", mapping_hc_path)


# =========================
# 5. mobility 1개 로드 + 집계
# =========================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.columns.tolist())
print(mobility.head())

needed = ["datetime", "CELL_ID_BASE", "out_cnt", "in_cnt", "net_cnt"]
missing = [c for c in needed if c not in mobility.columns]
if missing:
    raise ValueError(f"mobility 컬럼 없음: {missing}")

mobility = mobility[needed].copy()

mobility = mobility.rename(columns={
    "CELL_ID_BASE": "CELL_ID",
    "out_cnt": "mobility_out_cnt",
    "in_cnt": "mobility_in_cnt",
    "net_cnt": "mobility_net_cnt"
})

for c in ["mobility_out_cnt", "mobility_in_cnt", "mobility_net_cnt"]:
    mobility[c] = pd.to_numeric(mobility[c], errors="coerce").fillna(0)

cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        mobility_out_sum=("mobility_out_cnt", "sum"),
        mobility_in_sum=("mobility_in_cnt", "sum"),
        mobility_net_sum=("mobility_net_cnt", "sum"),
        mobility_out_mean=("mobility_out_cnt", "mean"),
        mobility_in_mean=("mobility_in_cnt", "mean"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# =========================
# 6. station activity 집계
# =========================

for col in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[col] = pd.to_numeric(station_30[col], errors="coerce").fillna(0)
    station_daily[col] = pd.to_numeric(station_daily[col], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] + station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] + station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())


# =========================
# 7. 최종 merge
# =========================

final = cell_mobility.merge(
    mapping_hc,
    on="CELL_ID",
    how="inner"
)

final = final.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="left"
)

activity_cols = [c for c in final.columns if c.startswith("station_")]
for c in activity_cols:
    if final[c].dtype != "object":
        final[c] = final[c].fillna(0)

print("\n[FINAL MERGE CHECK]")
print("mobility cells:", cell_mobility["CELL_ID"].nunique())
print("high-confidence mapping cells:", mapping_hc["CELL_ID"].nunique())
print("final cells:", final["CELL_ID"].nunique())
print("final rows:", len(final))
print("activity matched rows:", (final["station_total_activity"] > 0).sum())
print("activity zero rows:", (final["station_total_activity"] == 0).sum())


# =========================
# 8. 연구 지표
# =========================

final["actual_total_flow_norm"] = normalize_minmax(final["actual_total_flow"])
final["station_activity_norm"] = normalize_minmax(final["station_total_activity"])
final["accessibility_norm"] = normalize_minmax(final["station_accessibility_score"])
final["distance_norm"] = normalize_minmax(final["nearest_station_distance_m"])

final["hidden_demand_score"] = (
    final["actual_total_flow_norm"] *
    (1 - final["accessibility_norm"])
)

final["subway_representation_gap"] = (
    final["actual_total_flow_norm"] -
    final["station_activity_norm"]
)

final["subway_overrepresentation_score"] = (
    final["station_activity_norm"] *
    (1 - final["actual_total_flow_norm"])
)

final["station_area_mobility_score"] = (
    final["actual_total_flow_norm"] *
    final["accessibility_norm"]
)

q_hidden = final["hidden_demand_score"].quantile(0.90)
q_gap = final["subway_representation_gap"].quantile(0.90)
q_station_area = final["station_area_mobility_score"].quantile(0.90)

conditions = [
    final["hidden_demand_score"] >= q_hidden,
    final["subway_representation_gap"] >= q_gap,
    final["station_area_mobility_score"] >= q_station_area
]

choices = [
    "hidden_demand_area",
    "subway_under_represented",
    "station_area_high_mobility"
]

final["area_type"] = np.select(conditions, choices, default="normal")


# =========================
# 9. 요약 저장
# =========================

top_hidden = (
    final
    .sort_values("hidden_demand_score", ascending=False)
    .head(200)
)

station_summary = (
    final
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        cell_count=("CELL_ID", "count"),
        total_actual_flow=("actual_total_flow", "sum"),
        mean_actual_flow=("actual_total_flow", "mean"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        station_total_activity=("station_total_activity", "mean"),
        mean_hidden_demand=("hidden_demand_score", "mean"),
        max_hidden_demand=("hidden_demand_score", "max"),
        mean_representation_gap=("subway_representation_gap", "mean"),
        hidden_cell_count=("area_type", lambda x: (x == "hidden_demand_area").sum())
    )
)

station_summary["hidden_cell_ratio"] = (
    station_summary["hidden_cell_count"] / station_summary["cell_count"]
)

station_summary = station_summary.sort_values(
    ["hidden_cell_count", "total_actual_flow"],
    ascending=False
)

final.to_csv(final_path, index=False, encoding="utf-8-sig")
top_hidden.to_csv(top_hidden_path, index=False, encoding="utf-8-sig")
station_summary.to_csv(station_summary_path, index=False, encoding="utf-8-sig")


# =========================
# 10. 결과 확인
# =========================

print("\n================================================")
print("DONE")
print("================================================")
print("dict clean:", dict_clean_path)
print("mapping high confidence:", mapping_hc_path)
print("final:", final_path)
print("top hidden:", top_hidden_path)
print("station summary:", station_summary_path)

print("\n[FINAL SHAPE]")
print(final.shape)

print("\n[KEY METRICS]")
print(final[[
    "actual_total_flow",
    "nearest_station_distance_m",
    "station_total_activity",
    "actual_total_flow_norm",
    "station_activity_norm",
    "accessibility_norm",
    "hidden_demand_score",
    "subway_representation_gap",
    "subway_overrepresentation_score",
    "station_area_mobility_score"
]].describe())

print("\n[AREA TYPE COUNTS]")
print(final["area_type"].value_counts())

print("\n[TOP HIDDEN]")
print(top_hidden[[
    "CELL_ID",
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "nearest_station",
    "nearest_line",
    "actual_total_flow",
    "nearest_station_distance_m",
    "station_total_activity",
    "hidden_demand_score",
    "subway_representation_gap",
    "area_type"
]].head(30))

print("\n[STATION SUMMARY TOP]")
print(station_summary.head(30))

read success: 서울교통공사_역명 지하철역 검색.csv / cp949

[DICT RAW]
(799, 4)
['전철역코드', '전철역명', '호선', '외부코드']
  전철역코드       전철역명    호선  외부코드
0  101C         회기   경의선  K118
1  0205  동대문역사문화공원  02호선   205
2  0206         신당  02호선   206
3  0207       상왕십리  02호선   207
4  0208        왕십리  02호선   208

[DICT CLEAN]
(426, 6)
   STATS_STATION_ID station_name station_name_clean dict_line  \
1               205    동대문역사문화공원          동대문역사문화공원      02호선   
2               206           신당                 신당      02호선   
3               207         상왕십리               상왕십리      02호선   
4               208          왕십리                왕십리      02호선   
5               209          한양대                한양대      02호선   

  subway_station_code external_code_raw  
1                0205               205  
2                0206               206  
3                0207               207  
4                0208               208  
5                0209               209  
saved: C:\Users\82108\Desktop\새 폴더 (8)\research_hig

In [ ]:
import pandas as pd

# CELL master
cell_master = pd.read_csv(
    r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv",
    encoding="utf-8-sig"
)

# mobility parquet
mobility = pd.read_parquet(
    r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"
)

print("\n[CELL MASTER]")
print(cell_master.head())

print("\n[MOBILITY]")
print(mobility.head())

# 예시 index 확인
idx = 26

print("\n[INDEX TEST]")
print(cell_master.iloc[idx])


[MAPPING]
(207205, 19)
['CELL_ID', 'CELL_X', 'CELL_Y', 'STATION_ID', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'station_accessibility_score', 'is_station_area_5km', 'valid_nearest_station']
      CELL_ID  CELL_X   CELL_Y  STATION_ID nearest_station nearest_line  \
0  나사99757775  899875  1977875        4920              양촌       김포골드라인   
1  다사18001325  918125  1913375        3135          지식정보단지        인천1호선   
2  다사31254225  931375  1942375        3121              동수        인천1호선   
3  다사33752875  933875  1928875        1761              정왕          안산선   
4  다바40009800  940125  1898125        1875              어천          수인선   

   station_lat  station_lon  station_x_5179  station_y_5179  \
0    37.642379   126.614309   921861.795997    1.960691e+06   
1    37.378384   126.645168   9

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로
# ============================================================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_level_research_test"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"
station_dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"
station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

cell_mobility_output = os.path.join(output_dir, "01_cell_mobility_restored.csv")
station_mobility_output = os.path.join(output_dir, "02_station_surrounding_mobility.csv")
final_output = os.path.join(output_dir, "03_station_level_final_gap_analysis.csv")
top_hidden_output = os.path.join(output_dir, "04_top_hidden_demand_stations.csv")

# ============================================================
# 1. 유틸
# ============================================================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")

def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def clean_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )

# ============================================================
# 2. CELL master 로드
# ============================================================

cell_master = read_csv_safe(cell_master_path)
cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()

print("\n[CELL MASTER]")
print(cell_master.shape)
print(cell_master.head())

# ============================================================
# 3. mobility 로드 + CELL_ID_BASE → 실제 CELL_ID 복원
# ============================================================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.head())

mobility["CELL_INDEX"] = pd.to_numeric(mobility["CELL_ID_BASE"], errors="coerce")
mobility = mobility.dropna(subset=["CELL_INDEX"]).copy()
mobility["CELL_INDEX"] = mobility["CELL_INDEX"].astype(int)

valid_idx = (
    (mobility["CELL_INDEX"] >= 0) &
    (mobility["CELL_INDEX"] < len(cell_master))
)

print("\n[CELL INDEX CHECK]")
print("valid ratio:", valid_idx.mean())
print("min:", mobility["CELL_INDEX"].min())
print("max:", mobility["CELL_INDEX"].max())
print("cell master rows:", len(cell_master))

mobility = mobility[valid_idx].copy().reset_index(drop=True)

matched_cell = cell_master.iloc[mobility["CELL_INDEX"].values].reset_index(drop=True)

mobility["CELL_ID"] = matched_cell["CELL_ID"].values
mobility["CELL_X"] = matched_cell["CELL_X"].values
mobility["CELL_Y"] = matched_cell["CELL_Y"].values

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility[c] = pd.to_numeric(mobility[c], errors="coerce").fillna(0)

# CELL 단위 생활이동량 집계
cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_out_mean=("out_cnt", "mean"),
        mobility_in_mean=("in_cnt", "mean"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

cell_mobility.to_csv(cell_mobility_output, index=False, encoding="utf-8-sig")

print("\n[CELL MOBILITY RESTORED]")
print(cell_mobility.shape)
print(cell_mobility.head())

# ============================================================
# 4. cell → nearest station mapping 로드
# ============================================================

mapping = read_csv_safe(mapping_path)

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

mapping["nearest_station_clean"] = clean_name(mapping["nearest_station"])

print("\n[MAPPING]")
print(mapping.shape)
print(mapping.head())

# ============================================================
# 5. station dictionary 로드
# ============================================================

station_dict = read_csv_safe(station_dict_path)

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_name(station_dict["station_name"])

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

print("\n[STATION DICT]")
print(station_dict.shape)
print(station_dict.head())

# ============================================================
# 6. station activity 로드 + valid dictionary 생성
# ============================================================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

station_30["STATION_ID"] = pd.to_numeric(station_30["STATION_ID"], errors="coerce")
station_daily["STATION_ID"] = pd.to_numeric(station_daily["STATION_ID"], errors="coerce")

activity_ids = (
    set(station_30["STATION_ID"].dropna().astype(int).unique()) |
    set(station_daily["STATION_ID"].dropna().astype(int).unique())
)

station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[VALID STATION DICT]")
print(station_dict_valid.shape)
print("unique STATS_STATION_ID:", station_dict_valid["STATS_STATION_ID"].nunique())

# mapping에 STATS_STATION_ID 붙이기
mapping_hc = mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[HIGH CONFIDENCE CELL-STATION MAPPING]")
print("mapping rows:", len(mapping))
print("hc rows:", len(mapping_hc))
print("unique STATS_STATION_ID:", mapping_hc["STATS_STATION_ID"].nunique())

# ============================================================
# 7. 핵심 수정: CELL 단위가 아니라 STATION 단위로 주변 mobility 집계
# ============================================================

cell_with_station = cell_mobility.merge(
    mapping_hc,
    on="CELL_ID",
    how="inner"
)

print("\n[CELL MOBILITY + STATION MAPPING]")
print(cell_with_station.shape)
print(cell_with_station.head())

station_surrounding_mobility = (
    cell_with_station
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        surrounding_cell_count=("CELL_ID", "count"),
        surrounding_total_flow=("actual_total_flow", "sum"),
        surrounding_outflow=("mobility_out_sum", "sum"),
        surrounding_inflow=("mobility_in_sum", "sum"),
        surrounding_net_flow=("mobility_net_sum", "sum"),
        surrounding_abs_net_flow=("actual_abs_net_flow", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

station_surrounding_mobility.to_csv(
    station_mobility_output,
    index=False,
    encoding="utf-8-sig"
)

print("\n[STATION SURROUNDING MOBILITY]")
print(station_surrounding_mobility.shape)
print(station_surrounding_mobility.head())

# ============================================================
# 8. station activity 집계
# ============================================================

for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[c] = pd.to_numeric(station_30[c], errors="coerce").fillna(0)
    station_daily[c] = pd.to_numeric(station_daily[c], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] +
    station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] +
    station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())

# ============================================================
# 9. 최종: 역 주변 생활이동량 vs 지하철 activity 비교
# ============================================================

final_station = station_surrounding_mobility.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="inner"
)

print("\n[FINAL STATION LEVEL MERGE]")
print(final_station.shape)
print(final_station.head())

# ============================================================
# 10. 연구 지표 생성
# ============================================================

final_station["surrounding_total_flow_norm"] = normalize_minmax(
    final_station["surrounding_total_flow"]
)

final_station["station_activity_norm"] = normalize_minmax(
    final_station["station_total_activity"]
)

final_station["distance_norm"] = normalize_minmax(
    final_station["mean_distance_m"]
)

# 핵심 1: 주변 생활이동은 큰데 지하철 activity는 낮은 역
final_station["subway_representation_gap"] = (
    final_station["surrounding_total_flow_norm"] -
    final_station["station_activity_norm"]
)

# 핵심 2: hidden demand
final_station["hidden_demand_score"] = (
    final_station["sur

read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[CELL MASTER]
(207205, 3)
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125

[MOBILITY RAW]
(630941, 5)
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt
0 2023-01-02           26      7.0     0.0     -7.0
1 2023-01-02           27     10.5     0.0    -10.5
2 2023-01-02           29     21.0     0.0    -21.0
3 2023-01-02           30     17.5     7.0    -10.5
4 2023-01-02        42110     10.5     0.0    -10.5

[CELL INDEX CHECK]
valid ratio: 1.0
min: 26
max: 44825
cell master rows: 207205

[MOBILITY RESTORED]
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt  CELL_INDEX     CELL_ID  \
0 2023-01-02           26      7.0     0.0     -7.0          26  다사62256100   
1 2023-01-02           27     10.5     0.0    -10.5          27  다사70256000   
2 2023-01-02           29     21.0     0.0    -21

In [9]:
# ============================================================
# STATION-LEVEL RESEARCH PIPELINE TEST
# 목적:
# 지하철역 주변 cell 생활이동량 집계
# vs 역별 subway activity 비교
#
# 데이터 1개씩만 사용
# ============================================================

import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로
# ============================================================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_level_research_test"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"

mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"

station_dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"

station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

station_level_output_path = os.path.join(output_dir, "STATION_LEVEL_TEST_dataset.csv")
top_gap_output_path = os.path.join(output_dir, "STATION_LEVEL_TEST_top_gap.csv")
cell_join_output_path = os.path.join(output_dir, "CELL_JOIN_TEST_debug.csv")


# ============================================================
# 1. 유틸
# ============================================================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


def clean_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )


# ============================================================
# 2. CELL master 로드
# ============================================================

cell_master = read_csv_safe(cell_master_path)

cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()
cell_master["CELL_X"] = pd.to_numeric(cell_master["CELL_X"], errors="coerce")
cell_master["CELL_Y"] = pd.to_numeric(cell_master["CELL_Y"], errors="coerce")

print("\n[CELL MASTER]")
print(cell_master.shape)
print(cell_master.head())


# ============================================================
# 3. mobility 로드 + CELL_ID_BASE -> 실제 CELL_ID 복원
# ============================================================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.head())

mobility["CELL_INDEX"] = pd.to_numeric(
    mobility["CELL_ID_BASE"],
    errors="coerce"
)

mobility = mobility.dropna(subset=["CELL_INDEX"]).copy()
mobility["CELL_INDEX"] = mobility["CELL_INDEX"].astype(int)

valid_idx = (
    (mobility["CELL_INDEX"] >= 0) &
    (mobility["CELL_INDEX"] < len(cell_master))
)

print("\n[CELL INDEX CHECK]")
print("valid ratio:", valid_idx.mean())
print("min:", mobility["CELL_INDEX"].min())
print("max:", mobility["CELL_INDEX"].max())
print("cell master rows:", len(cell_master))

mobility = mobility[valid_idx].copy().reset_index(drop=True)

matched_cell = cell_master.iloc[mobility["CELL_INDEX"].values].reset_index(drop=True)

mobility["CELL_ID"] = matched_cell["CELL_ID"].values
mobility["CELL_X"] = matched_cell["CELL_X"].values
mobility["CELL_Y"] = matched_cell["CELL_Y"].values

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility[c] = pd.to_numeric(mobility[c], errors="coerce").fillna(0)

print("\n[MOBILITY RESTORED]")
print(mobility.head())


# ============================================================
# 4. CELL 단위 mobility 집계
# ============================================================

cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] +
    cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# ============================================================
# 5. cell -> nearest station mapping 로드
# ============================================================

mapping = read_csv_safe(mapping_path)

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

mapping["nearest_station_clean"] = clean_name(mapping["nearest_station"])

print("\n[MAPPING]")
print(mapping.shape)
print(mapping.head())


# ============================================================
# 6. station dictionary 로드
#    서울교통공사_역명 지하철역 검색.csv = ID bridge
# ============================================================

station_dict = read_csv_safe(station_dict_path)

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_name(station_dict["station_name"])

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

print("\n[STATION DICT]")
print(station_dict.shape)
print(station_dict.head())


# ============================================================
# 7. station activity 로드 + station id 존재하는 dictionary만 사용
# ============================================================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

station_30["STATION_ID"] = pd.to_numeric(station_30["STATION_ID"], errors="coerce")
station_daily["STATION_ID"] = pd.to_numeric(station_daily["STATION_ID"], errors="coerce")

activity_ids = (
    set(station_30["STATION_ID"].dropna().astype(int).unique())
    |
    set(station_daily["STATION_ID"].dropna().astype(int).unique())
)

station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[VALID DICT]")
print(station_dict_valid.shape)
print("unique STATS_STATION_ID:", station_dict_valid["STATS_STATION_ID"].nunique())


# ============================================================
# 8. mapping에 STATS_STATION_ID 부여
#    unmatched는 버림
# ============================================================

mapping_hc = mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[HIGH CONFIDENCE MAPPING]")
print("mapping rows:", len(mapping))
print("hc rows:", len(mapping_hc))
print("kept ratio:", len(mapping_hc) / len(mapping))
print("unique stats station:", mapping_hc["STATS_STATION_ID"].nunique())


# ============================================================
# 9. 핵심: CELL mobility + mapping 결합
#    그 다음 STATION 기준으로 주변 mobility 집계
# ============================================================

cell_join = mapping_hc.merge(
    cell_mobility,
    on="CELL_ID",
    how="left",
    suffixes=("", "_mobility")
)

# mobility 없는 cell은 0으로 둠
for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "mobility_record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_join[c] = pd.to_numeric(cell_join[c], errors="coerce").fillna(0)

print("\n[CELL JOIN]")
print("mapping_hc cells:", mapping_hc["CELL_ID"].nunique())
print("cell_mobility cells:", cell_mobility["CELL_ID"].nunique())
print("joined rows:", len(cell_join))
print("cells with mobility > 0:", (cell_join["actual_total_flow"] > 0).sum())
print(cell_join[[
    "CELL_ID",
    "STATS_STATION_ID",
    "station_name",
    "nearest_station",
    "nearest_station_distance_m",
    "actual_total_flow"
]].head())

cell_join.to_csv(cell_join_output_path, index=False, encoding="utf-8-sig")


# station 주변 mobility aggregate
station_mobility = (
    cell_join
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        nearby_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        station_area_mobility_total=("actual_total_flow", "sum"),
        station_area_out_total=("mobility_out_sum", "sum"),
        station_area_in_total=("mobility_in_sum", "sum"),
        station_area_net_total=("mobility_net_sum", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

print("\n[STATION MOBILITY]")
print(station_mobility.shape)
print(station_mobility.head())


# ============================================================
# 10. station activity 집계
# ============================================================

for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[c] = pd.to_numeric(station_30[c], errors="coerce").fillna(0)
    station_daily[c] = pd.to_numeric(station_daily[c], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] +
    station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] +
    station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())


# ============================================================
# 11. 최종 station-level merge
# ============================================================

station_level = station_mobility.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="left"
)

for c in station_level.columns:
    if c.startswith("station_") and station_level[c].dtype != "object":
        station_level[c] = station_level[c].fillna(0)

print("\n[STATION LEVEL MERGE]")
print(station_level.shape)
print(station_level.head())


# ============================================================
# 12. 연구 지표 계산
# ============================================================

station_level["mobility_norm"] = normalize_minmax(
    station_level["station_area_mobility_total"]
)

station_level["subway_activity_norm"] = normalize_minmax(
    station_level["station_total_activity"]
)

station_level["mean_distance_norm"] = normalize_minmax(
    station_level["mean_distance_m"]
)

# 실제 주변 이동량은 높은데, 지하철 activity는 낮은 역
station_level["subway_under_representation_gap"] = (
    station_level["mobility_norm"] -
    station_level["subway_activity_norm"]
)

# 지하철 activity는 높은데, 주변 생활이동량은 낮은 역
station_level["subway_over_representation_score"] = (
    station_level["subway_activity_norm"] *
    (1 - station_level["mobility_norm"])
)

# 역 주변 mobility 자체가 높은 곳
station_level["hidden_demand_score"] = (
    station_level["mobility_norm"] *
    (1 - station_level["subway_activity_norm"])
)

# 역 주변 cell은 많고 실제 이동도 큰 곳
station_level["station_area_demand_score"] = (
    station_level["mobility_norm"] *
    normalize_minmax(station_level["active_cell_count"])
)

q_hidden = station_level["hidden_demand_score"].quantile(0.90)
q_gap = station_level["subway_under_representation_gap"].quantile(0.90)
q_over = station_level["subway_over_representation_score"].quantile(0.90)

conditions = [
    station_level["hidden_demand_score"] >= q_hidden,
    station_level["subway_under_representation_gap"] >= q_gap,
    station_level["subway_over_representation_score"] >= q_over
]

choices = [
    "hidden_demand_station",
    "subway_under_represented",
    "subway_over_represented"
]

station_level["station_type"] = np.select(
    conditions,
    choices,
    default="normal"
)

top_gap = (
    station_level
    .sort_values("hidden_demand_score", ascending=False)
    .head(100)
)

# ============================================================
# 13. 저장
# ============================================================

station_level.to_csv(
    station_level_output_path,
    index=False,
    encoding="utf-8-sig"
)

top_gap.to_csv(
    top_gap_output_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 14. 결과 확인
# ============================================================

print("\n================================================")
print("DONE")
print("================================================")
print("station level:", station_level_output_path)
print("top gap:", top_gap_output_path)
print("cell join debug:", cell_join_output_path)

print("\n[FINAL STATION LEVEL SHAPE]")
print(station_level.shape)

print("\n[KEY METRICS]")
print(station_level[[
    "station_area_mobility_total",
    "station_total_activity",
    "mobility_norm",
    "subway_activity_norm",
    "subway_under_representation_gap",
    "subway_over_representation_score",
    "hidden_demand_score",
    "nearby_cell_count",
    "active_cell_count",
    "mean_distance_m"
]].describe())

print("\n[STATION TYPE COUNTS]")
print(station_level["station_type"].value_counts())

print("\n[TOP HIDDEN / GAP STATIONS]")
print(top_gap[[
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "nearby_cell_count",
    "active_cell_count",
    "station_area_mobility_total",
    "station_total_activity",
    "mobility_norm",
    "subway_activity_norm",
    "hidden_demand_score",
    "subway_under_representation_gap",
    "station_type"
]].head(30))

read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[CELL MASTER]
(207205, 3)
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125

[MOBILITY RAW]
(630941, 5)
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt
0 2023-01-02           26      7.0     0.0     -7.0
1 2023-01-02           27     10.5     0.0    -10.5
2 2023-01-02           29     21.0     0.0    -21.0
3 2023-01-02           30     17.5     7.0    -10.5
4 2023-01-02        42110     10.5     0.0    -10.5

[CELL INDEX CHECK]
valid ratio: 1.0
min: 26
max: 44825
cell master rows: 207205

[MOBILITY RESTORED]
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt  CELL_INDEX     CELL_ID  \
0 2023-01-02           26      7.0     0.0     -7.0          26  다사62256100   
1 2023-01-02           27     10.5     0.0    -10.5          27  다사70256000   
2 2023-01-02           29     21.0     0.0    -21

In [10]:
import os
import glob
import pandas as pd

folder = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output"

files = sorted(glob.glob(os.path.join(folder, "*.parquet")))

print("parquet files:", len(files))

summary = []

for i, path in enumerate(files):
    try:
        df = pd.read_parquet(path)

        row = {
            "file": os.path.basename(path),
            "rows": len(df),
            "cols": len(df.columns),
            "columns": ", ".join(df.columns.astype(str)),
        }

        if "CELL_ID_BASE" in df.columns:
            row["cell_unique"] = df["CELL_ID_BASE"].nunique()
            row["cell_min"] = pd.to_numeric(df["CELL_ID_BASE"], errors="coerce").min()
            row["cell_max"] = pd.to_numeric(df["CELL_ID_BASE"], errors="coerce").max()

        if "datetime" in df.columns:
            row["date_min"] = df["datetime"].min()
            row["date_max"] = df["datetime"].max()

        for c in ["out_cnt", "in_cnt", "net_cnt"]:
            if c in df.columns:
                row[f"{c}_sum"] = pd.to_numeric(df[c], errors="coerce").sum()

        summary.append(row)

        if (i + 1) % 50 == 0:
            print(f"checked {i+1}/{len(files)}")

    except Exception as e:
        summary.append({
            "file": os.path.basename(path),
            "error": str(e)
        })

summary_df = pd.DataFrame(summary)

out_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_parquet_structure_summary.csv"
summary_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("\n[DONE]")
print("saved:", out_path)

print("\n[HEAD]")
print(summary_df.head(20))

print("\n[LARGEST CELL UNIQUE]")
print(
    summary_df
    .sort_values("cell_unique", ascending=False)
    .head(20)[["file", "rows", "cell_unique", "out_cnt_sum", "in_cnt_sum", "date_min", "date_max"]]
)

parquet files: 380


KeyboardInterrupt: 

In [11]:
path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

df = pd.read_parquet(path)

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head(20))
print(df.describe(include="all"))

(630941, 5)
['datetime', 'CELL_ID_BASE', 'out_cnt', 'in_cnt', 'net_cnt']
datetime        datetime64[ns]
CELL_ID_BASE    string[python]
out_cnt                float64
in_cnt                 float64
net_cnt                float64
dtype: object
     datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt
0  2023-01-02           26      7.0     0.0     -7.0
1  2023-01-02           27     10.5     0.0    -10.5
2  2023-01-02           29     21.0     0.0    -21.0
3  2023-01-02           30     17.5     7.0    -10.5
4  2023-01-02        42110     10.5     0.0    -10.5
5  2023-01-02        42130     17.5     0.0    -17.5
6  2023-01-02        42150     21.0     3.5    -17.5
7  2023-01-02        42210     17.5     0.0    -17.5
8  2023-01-02        42720     28.0     3.5    -24.5
9  2023-01-02        42730      3.5     0.0     -3.5
10 2023-01-02        42760     10.5     0.0    -10.5
11 2023-01-02        42770      7.0     0.0     -7.0
12 2023-01-02        42820      7.0     0.0     -7.0
13 2023-01-02   

In [12]:
import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로
# =========================
output_dir = r"C:\Users\82108\Desktop\sample_data_check"
os.makedirs(output_dir, exist_ok=True)

paths = {
    "cell_master": r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv",
    "station_mapping": r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv",
    "station_dict": r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv",
    "mobility_cell": r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet",
    "station_30min": r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet",
    "station_daily": r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet",
}

# =========================
# 1. 읽기 함수
# =========================
def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")

def read_any(path):
    if path.lower().endswith(".parquet"):
        return pd.read_parquet(path)
    return read_csv_safe(path)

# =========================
# 2. 기본 구조 확인
# =========================
def inspect_df(name, df):
    print("\n" + "=" * 90)
    print(f"[DATASET] {name}")
    print("=" * 90)

    print("\n[SHAPE]")
    print(df.shape)

    print("\n[COLUMNS]")
    print(df.columns.tolist())

    print("\n[DTYPES]")
    print(df.dtypes)

    print("\n[HEAD]")
    print(df.head(10))

    print("\n[NULL COUNT]")
    print(df.isnull().sum())

    print("\n[MEMORY]")
    mem = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"{mem:.2f} MB")

    preview_path = os.path.join(output_dir, f"{name}_preview.csv")
    df.head(1000).to_csv(preview_path, index=False, encoding="utf-8-sig")
    print("\n[SAVED PREVIEW]")
    print(preview_path)

# =========================
# 3. 하나씩 읽고 확인
# =========================
loaded = {}

for name, path in paths.items():
    print("\n\nLOADING:", name)
    df = read_any(path)
    loaded[name] = df
    inspect_df(name, df)

# =========================
# 4. mobility CELL_ID_BASE 복원 가능성 확인
# =========================
print("\n" + "=" * 90)
print("[CHECK] mobility CELL_ID_BASE -> cell_master index")
print("=" * 90)

cell_master = loaded["cell_master"].copy()
mobility = loaded["mobility_cell"].copy()

cell_master = cell_master.reset_index().rename(columns={"index": "CELL_INDEX"})

mobility["CELL_INDEX"] = pd.to_numeric(
    mobility["CELL_ID_BASE"],
    errors="coerce"
)

valid_idx = (
    mobility["CELL_INDEX"].notna()
    & (mobility["CELL_INDEX"] >= 0)
    & (mobility["CELL_INDEX"] < len(cell_master))
)

print("mobility rows:", len(mobility))
print("CELL_ID_BASE unique:", mobility["CELL_ID_BASE"].nunique())
print("CELL_INDEX min:", mobility["CELL_INDEX"].min())
print("CELL_INDEX max:", mobility["CELL_INDEX"].max())
print("cell_master rows:", len(cell_master))
print("valid index ratio:", valid_idx.mean())

sample_idx = mobility.loc[valid_idx, "CELL_INDEX"].astype(int).drop_duplicates().head(20)

print("\n[SAMPLE INDEX -> CELL_ID]")
for idx in sample_idx:
    row = cell_master.loc[cell_master["CELL_INDEX"] == idx].iloc[0]
    print(idx, "->", row["CELL_ID"], row["CELL_X"], row["CELL_Y"])

# =========================
# 5. 안전한 방식으로 CELL_ID 복원 테스트
# =========================
mobility_restore_test = mobility.copy()
mobility_restore_test = mobility_restore_test[valid_idx].copy()
mobility_restore_test["CELL_INDEX"] = mobility_restore_test["CELL_INDEX"].astype(int)

mobility_restore_test = mobility_restore_test.merge(
    cell_master[["CELL_INDEX", "CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_INDEX",
    how="left"
)

print("\n[RESTORE TEST]")
print("restored rows:", len(mobility_restore_test))
print("restored CELL_ID unique:", mobility_restore_test["CELL_ID"].nunique())
print(mobility_restore_test.head(20))

restore_preview_path = os.path.join(output_dir, "mobility_restored_preview.csv")
mobility_restore_test.head(5000).to_csv(
    restore_preview_path,
    index=False,
    encoding="utf-8-sig"
)
print("\n[SAVED RESTORE PREVIEW]")
print(restore_preview_path)

# =========================
# 6. station id 겹침 확인
# =========================
print("\n" + "=" * 90)
print("[CHECK] station id overlap")
print("=" * 90)

station_30 = loaded["station_30min"].copy()
station_daily = loaded["station_daily"].copy()
station_dict = loaded["station_dict"].copy()

station_30_ids = set(pd.to_numeric(station_30["STATION_ID"], errors="coerce").dropna().astype(int))
station_daily_ids = set(pd.to_numeric(station_daily["STATION_ID"], errors="coerce").dropna().astype(int))

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "line",
    "외부코드": "external_code"
})

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

dict_ids = set(station_dict["STATS_STATION_ID"].dropna().astype(int))

print("30min ids:", len(station_30_ids))
print("daily ids:", len(station_daily_ids))
print("dict numeric ids:", len(dict_ids))
print("30min ∩ daily:", len(station_30_ids & station_daily_ids))
print("dict ∩ 30min:", len(dict_ids & station_30_ids))
print("dict ∩ daily:", len(dict_ids & station_daily_ids))

print("\n[OVERLAP SAMPLE]")
print(sorted(list(dict_ids & station_30_ids))[:50])

# =========================
# 7. mapping과 restored mobility overlap 확인
# =========================
print("\n" + "=" * 90)
print("[CHECK] restored mobility CELL_ID ∩ station mapping CELL_ID")
print("=" * 90)

mapping = loaded["station_mapping"].copy()

mobility_cell_ids = set(mobility_restore_test["CELL_ID"].dropna().astype(str))
mapping_cell_ids = set(mapping["CELL_ID"].dropna().astype(str))

print("restored mobility unique CELL_ID:", len(mobility_cell_ids))
print("station mapping unique CELL_ID:", len(mapping_cell_ids))
print("overlap:", len(mobility_cell_ids & mapping_cell_ids))

print("\n[OVERLAP SAMPLE]")
print(list(mobility_cell_ids & mapping_cell_ids)[:30])

print("\nDONE")
print("output folder:", output_dir)



LOADING: cell_master
read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[DATASET] cell_master

[SHAPE]
(207205, 3)

[COLUMNS]
['CELL_ID', 'CELL_X', 'CELL_Y']

[DTYPES]
CELL_ID    object
CELL_X      int64
CELL_Y      int64
dtype: object

[HEAD]
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125
5  다사53005725  953125  1957375
6  다사53255000  953375  1950125
7  다아53752450  953875  2024625
8  다아58500350  958625  2003625
9  다사59254775  959375  1947875

[NULL COUNT]
CELL_ID    0
CELL_X     0
CELL_Y     0
dtype: int64

[MEMORY]
20.16 MB

[SAVED PREVIEW]
C:\Users\82108\Desktop\sample_data_check\cell_master_preview.csv


LOADING: station_mapping
read success: cell_to_nearest_station_VALID_5km.csv / utf-8-sig

[DATASET] station_mapping

[SHAPE]
(85543, 19)

[COLUMNS]
['CELL_ID', 'CELL_X', 'CELL_Y', 'STATION_ID', 'nearest_station', 'nearest_line', 'stati

In [14]:
import os
import pandas as pd
import numpy as np

# ============================================================
# STATION-LEVEL RESEARCH PIPELINE
# CELL_ID_BASE 혼합형 복원 반영 완료
# ============================================================

# =========================
# 0. 경로
# =========================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_level_research_final_test"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"
station_dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"
station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

cell_mobility_output = os.path.join(output_dir, "01_cell_mobility_restored.csv")
station_mobility_output = os.path.join(output_dir, "02_station_surrounding_mobility.csv")
final_output = os.path.join(output_dir, "03_station_level_final_gap_analysis.csv")
top_hidden_output = os.path.join(output_dir, "04_top_hidden_demand_stations.csv")
debug_join_output = os.path.join(output_dir, "debug_cell_station_join.csv")


# =========================
# 1. 유틸
# =========================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


def clean_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )


# =========================
# 2. CELL master 로드
# =========================

cell_master = read_csv_safe(cell_master_path)

cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()
cell_master["CELL_ID"] = cell_master["CELL_ID"].astype(str).str.strip()
cell_master["CELL_X"] = pd.to_numeric(cell_master["CELL_X"], errors="coerce")
cell_master["CELL_Y"] = pd.to_numeric(cell_master["CELL_Y"], errors="coerce")

# index mapping용
cell_master_index = cell_master.reset_index().rename(columns={"index": "CELL_INDEX"})

print("\n[CELL MASTER]")
print(cell_master.shape)
print(cell_master.head())


# =========================
# 3. mobility 로드 + CELL_ID_BASE 혼합형 복원
# =========================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.head())

mobility["CELL_ID_BASE_STR"] = mobility["CELL_ID_BASE"].astype(str).str.strip()
mobility["CELL_INDEX"] = pd.to_numeric(mobility["CELL_ID_BASE_STR"], errors="coerce")

# 3-1. 숫자형 part: CELL master row index로 복원
numeric_part = mobility[mobility["CELL_INDEX"].notna()].copy()
numeric_part["CELL_INDEX"] = numeric_part["CELL_INDEX"].astype(int)

valid_numeric = (
    (numeric_part["CELL_INDEX"] >= 0) &
    (numeric_part["CELL_INDEX"] < len(cell_master_index))
)

numeric_part = numeric_part[valid_numeric].copy()

numeric_part = numeric_part.merge(
    cell_master_index[["CELL_INDEX", "CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_INDEX",
    how="left"
)

# 3-2. 문자형 part: 이미 실제 CELL_ID로 간주
string_part = mobility[mobility["CELL_INDEX"].isna()].copy()
string_part = string_part.rename(columns={"CELL_ID_BASE_STR": "CELL_ID"})
string_part["CELL_ID"] = string_part["CELL_ID"].astype(str).str.strip()

string_part = string_part.merge(
    cell_master[["CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_ID",
    how="left"
)

# 3-3. 합치기
mobility_restored = pd.concat(
    [numeric_part, string_part],
    ignore_index=True
)

# 좌표 매칭 실패 제거
before_rows = len(mobility_restored)
mobility_restored = mobility_restored.dropna(subset=["CELL_ID", "CELL_X", "CELL_Y"]).copy()
after_rows = len(mobility_restored)

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility_restored[c] = pd.to_numeric(mobility_restored[c], errors="coerce").fillna(0)

print("\n[MOBILITY RESTORE CHECK]")
print("original rows:", len(mobility))
print("numeric part rows:", len(numeric_part))
print("string part rows:", len(string_part))
print("restored rows before drop:", before_rows)
print("restored rows after drop:", after_rows)
print("restored unique CELL_ID:", mobility_restored["CELL_ID"].nunique())
print("missing CELL_ID:", mobility_restored["CELL_ID"].isna().sum())
print("missing CELL_X:", mobility_restored["CELL_X"].isna().sum())
print("missing CELL_Y:", mobility_restored["CELL_Y"].isna().sum())
print(mobility_restored[["datetime", "CELL_ID_BASE", "CELL_ID", "CELL_X", "CELL_Y", "out_cnt", "in_cnt", "net_cnt"]].head())


# =========================
# 4. CELL 단위 mobility 집계
# =========================

cell_mobility = (
    mobility_restored
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_out_mean=("out_cnt", "mean"),
        mobility_in_mean=("in_cnt", "mean"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

cell_mobility.to_csv(cell_mobility_output, index=False, encoding="utf-8-sig")

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# =========================
# 5. cell → nearest station mapping 로드
# =========================

mapping = read_csv_safe(mapping_path)

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

mapping["CELL_ID"] = mapping["CELL_ID"].astype(str).str.strip()
mapping["nearest_station_clean"] = clean_name(mapping["nearest_station"])

print("\n[MAPPING]")
print(mapping.shape)
print(mapping.head())


# =========================
# 6. station dictionary 로드
# =========================

station_dict = read_csv_safe(station_dict_path)

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_name(station_dict["station_name"])

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

print("\n[STATION DICT]")
print(station_dict.shape)
print(station_dict.head())


# =========================
# 7. station activity 로드 + valid dict
# =========================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

station_30["STATION_ID"] = pd.to_numeric(station_30["STATION_ID"], errors="coerce")
station_daily["STATION_ID"] = pd.to_numeric(station_daily["STATION_ID"], errors="coerce")

activity_ids = (
    set(station_30["STATION_ID"].dropna().astype(int).unique())
    |
    set(station_daily["STATION_ID"].dropna().astype(int).unique())
)

station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[VALID STATION DICT]")
print(station_dict_valid.shape)
print("unique STATS_STATION_ID:", station_dict_valid["STATS_STATION_ID"].nunique())


# =========================
# 8. mapping에 STATS_STATION_ID 부여
# =========================

mapping_hc = mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[HIGH CONFIDENCE CELL-STATION MAPPING]")
print("mapping rows:", len(mapping))
print("hc rows:", len(mapping_hc))
print("kept ratio:", len(mapping_hc) / len(mapping))
print("unique STATS_STATION_ID:", mapping_hc["STATS_STATION_ID"].nunique())


# =========================
# 9. CELL mobility + station mapping 결합
# =========================

cell_with_station = mapping_hc.merge(
    cell_mobility,
    on="CELL_ID",
    how="left",
    suffixes=("", "_mobility")
)

for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "mobility_out_mean",
    "mobility_in_mean",
    "mobility_record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_with_station[c] = pd.to_numeric(cell_with_station[c], errors="coerce").fillna(0)

cell_with_station.to_csv(debug_join_output, index=False, encoding="utf-8-sig")

print("\n[CELL WITH STATION CHECK]")
print("mapping_hc cells:", mapping_hc["CELL_ID"].nunique())
print("cell_mobility cells:", cell_mobility["CELL_ID"].nunique())
print("joined rows:", len(cell_with_station))
print("cells with mobility > 0:", (cell_with_station["actual_total_flow"] > 0).sum())
print("total surrounding mobility:", cell_with_station["actual_total_flow"].sum())
print(cell_with_station[[
    "CELL_ID",
    "STATS_STATION_ID",
    "station_name",
    "nearest_station",
    "nearest_station_distance_m",
    "actual_total_flow"
]].head())


# =========================
# 10. STATION 주변 mobility 집계
# =========================

station_surrounding_mobility = (
    cell_with_station
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        surrounding_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        surrounding_total_flow=("actual_total_flow", "sum"),
        surrounding_outflow=("mobility_out_sum", "sum"),
        surrounding_inflow=("mobility_in_sum", "sum"),
        surrounding_net_flow=("mobility_net_sum", "sum"),
        surrounding_abs_net_flow=("actual_abs_net_flow", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

station_surrounding_mobility.to_csv(
    station_mobility_output,
    index=False,
    encoding="utf-8-sig"
)

print("\n[STATION SURROUNDING MOBILITY]")
print(station_surrounding_mobility.shape)
print(station_surrounding_mobility.head())


# =========================
# 11. station activity 집계
# =========================

for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[c] = pd.to_numeric(station_30[c], errors="coerce").fillna(0)
    station_daily[c] = pd.to_numeric(station_daily[c], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] + station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] + station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] - station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())


# =========================
# 12. 최종 station-level merge
# =========================

final_station = station_surrounding_mobility.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="inner"
)

print("\n[FINAL STATION LEVEL MERGE]")
print(final_station.shape)
print(final_station.head())


# =========================
# 13. 연구 지표 생성
# =========================

final_station["surrounding_total_flow_norm"] = normalize_minmax(
    final_station["surrounding_total_flow"]
)

final_station["station_activity_norm"] = normalize_minmax(
    final_station["station_total_activity"]
)

final_station["distance_norm"] = normalize_minmax(
    final_station["mean_distance_m"]
)

final_station["subway_representation_gap"] = (
    final_station["surrounding_total_flow_norm"] -
    final_station["station_activity_norm"]
)

final_station["hidden_demand_score"] = (
    final_station["surrounding_total_flow_norm"] *
    (1 - final_station["station_activity_norm"])
)

final_station["distance_weighted_hidden_demand"] = (
    final_station["surrounding_total_flow_norm"] *
    final_station["distance_norm"]
)

final_station["subway_overrepresentation_score"] = (
    final_station["station_activity_norm"] *
    (1 - final_station["surrounding_total_flow_norm"])
)

q_hidden = final_station["hidden_demand_score"].quantile(0.90)
q_gap = final_station["subway_representation_gap"].quantile(0.90)
q_over = final_station["subway_overrepresentation_score"].quantile(0.90)

conditions = [
    final_station["hidden_demand_score"] >= q_hidden,
    final_station["subway_representation_gap"] >= q_gap,
    final_station["subway_overrepresentation_score"] >= q_over
]

choices = [
    "hidden_demand_station",
    "subway_under_represented",
    "subway_over_represented"
]

final_station["station_type"] = np.select(
    conditions,
    choices,
    default="normal"
)

final_station = final_station.sort_values(
    "hidden_demand_score",
    ascending=False
)

top_hidden_station = final_station.head(100)

# =========================
# 14. 저장
# =========================

final_station.to_csv(
    final_output,
    index=False,
    encoding="utf-8-sig"
)

top_hidden_station.to_csv(
    top_hidden_output,
    index=False,
    encoding="utf-8-sig"
)


# =========================
# 15. 결과 확인
# =========================

print("\n================================================")
print("DONE")
print("================================================")

print("cell mobility:", cell_mobility_output)
print("station surrounding mobility:", station_mobility_output)
print("debug join:", debug_join_output)
print("final station-level result:", final_output)
print("top hidden stations:", top_hidden_output)

print("\n[FINAL SHAPE]")
print(final_station.shape)

print("\n[KEY METRICS]")
print(final_station[[
    "surrounding_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "station_total_activity",
    "mean_distance_m",
    "surrounding_total_flow_norm",
    "station_activity_norm",
    "subway_representation_gap",
    "hidden_demand_score",
    "distance_weighted_hidden_demand",
    "subway_overrepresentation_score"
]].describe())

print("\n[STATION TYPE COUNTS]")
print(final_station["station_type"].value_counts())

print("\n[TOP HIDDEN DEMAND STATIONS]")
print(top_hidden_station[[
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "surrounding_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "station_total_activity",
    "mean_distance_m",
    "subway_representation_gap",
    "hidden_demand_score",
    "station_type"
]].head(30))

read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[CELL MASTER]
(207205, 3)
      CELL_ID  CELL_X   CELL_Y
0  나사99757775  899875  1977875
1  다사18001325  918125  1913375
2  다사31254225  931375  1942375
3  다사33752875  933875  1928875
4  다바40009800  940125  1898125

[MOBILITY RAW]
(630941, 5)
    datetime CELL_ID_BASE  out_cnt  in_cnt  net_cnt
0 2023-01-02           26      7.0     0.0     -7.0
1 2023-01-02           27     10.5     0.0    -10.5
2 2023-01-02           29     21.0     0.0    -21.0
3 2023-01-02           30     17.5     7.0    -10.5
4 2023-01-02        42110     10.5     0.0    -10.5

[MOBILITY RESTORE CHECK]
original rows: 630941
numeric part rows: 2067
string part rows: 628874
restored rows before drop: 630941
restored rows after drop: 630941
restored unique CELL_ID: 28475
missing CELL_ID: 0
missing CELL_X: 0
missing CELL_Y: 0
    datetime CELL_ID_BASE     CELL_ID  CELL_X   CELL_Y  out_cnt  in_cnt  \
0 2023-01-02           26  다사62256100  962375  1961125      7.0

In [15]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로
# ============================================================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\final_station_analysis_1file"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
station_mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"
station_dict_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"
station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

restored_mobility_path = os.path.join(output_dir, "01_restored_mobility_cell.csv")
cell_station_join_path = os.path.join(output_dir, "02_cell_mobility_station_join.csv")
station_result_path = os.path.join(output_dir, "03_station_level_result.csv")
top_hidden_path = os.path.join(output_dir, "04_top_hidden_demand_station.csv")


# ============================================================
# 1. 유틸
# ============================================================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def clean_name(s):
    return (
        s.astype(str)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


# ============================================================
# 2. CELL master 로드
# ============================================================

cell_master = read_csv_safe(cell_master_path)
cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()

cell_master = cell_master.reset_index().rename(columns={
    "index": "CELL_INDEX"
})

cell_master["CELL_INDEX"] = cell_master["CELL_INDEX"].astype(int)

print("\n[CELL MASTER]")
print(cell_master.shape)
print(cell_master.head())


# ============================================================
# 3. mobility 로드 + CELL_ID 복원
# ============================================================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print("CELL_ID_BASE unique:", mobility["CELL_ID_BASE"].nunique())

mobility["CELL_ID_BASE_RAW"] = mobility["CELL_ID_BASE"].astype(str).str.strip()

# 숫자만 있는 값 = cell_master index
is_numeric = mobility["CELL_ID_BASE_RAW"].str.fullmatch(r"\d+")

numeric_part = mobility[is_numeric].copy()
string_part = mobility[~is_numeric].copy()

numeric_part["CELL_INDEX"] = pd.to_numeric(
    numeric_part["CELL_ID_BASE_RAW"],
    errors="coerce"
).astype(int)

numeric_part = numeric_part.merge(
    cell_master[["CELL_INDEX", "CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_INDEX",
    how="left"
)

# 문자형은 이미 CELL_ID
string_part["CELL_ID"] = string_part["CELL_ID_BASE_RAW"]

string_part = string_part.merge(
    cell_master[["CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_ID",
    how="left"
)

mobility_restored = pd.concat(
    [numeric_part, string_part],
    ignore_index=True
)

print("\n[RESTORE CHECK]")
print("original rows:", len(mobility))
print("numeric rows:", len(numeric_part))
print("string rows:", len(string_part))
print("restored rows:", len(mobility_restored))
print("restored CELL_ID unique:", mobility_restored["CELL_ID"].nunique())
print("missing CELL_ID:", mobility_restored["CELL_ID"].isna().sum())
print("missing CELL_X:", mobility_restored["CELL_X"].isna().sum())

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility_restored[c] = pd.to_numeric(
        mobility_restored[c],
        errors="coerce"
    ).fillna(0)

# CELL 단위 mobility 집계
cell_mobility = (
    mobility_restored
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] +
    cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = (
    cell_mobility["mobility_net_sum"].abs()
)

cell_mobility.to_csv(
    restored_mobility_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# ============================================================
# 4. station mapping 로드
# ============================================================

station_mapping = read_csv_safe(station_mapping_path)

station_mapping = station_mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

station_mapping["nearest_station_clean"] = clean_name(
    station_mapping["nearest_station"]
)

print("\n[STATION MAPPING]")
print(station_mapping.shape)


# ============================================================
# 5. station dictionary 로드
# ============================================================

station_dict = read_csv_safe(station_dict_path)

station_dict = station_dict.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "dict_line",
    "외부코드": "external_code"
})

station_dict["station_name_clean"] = clean_name(station_dict["station_name"])

station_dict["STATS_STATION_ID"] = pd.to_numeric(
    station_dict["external_code"].astype(str).str.strip(),
    errors="coerce"
)

station_dict = station_dict.dropna(subset=["STATS_STATION_ID"]).copy()
station_dict["STATS_STATION_ID"] = station_dict["STATS_STATION_ID"].astype(int)

station_dict = station_dict[[
    "STATS_STATION_ID",
    "station_name",
    "station_name_clean",
    "dict_line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

print("\n[STATION DICT]")
print(station_dict.shape)


# ============================================================
# 6. station activity 로드 + 집계
# ============================================================

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

for df in [station_30, station_daily]:
    df["STATION_ID"] = pd.to_numeric(df["STATION_ID"], errors="coerce")
    for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

activity_ids = (
    set(station_30["STATION_ID"].dropna().astype(int).unique())
    |
    set(station_daily["STATION_ID"].dropna().astype(int).unique())
)

station_dict_valid = station_dict[
    station_dict["STATS_STATION_ID"].isin(activity_ids)
].copy()

print("\n[VALID DICT]")
print(station_dict_valid.shape)
print("unique STATS_STATION_ID:", station_dict_valid["STATS_STATION_ID"].nunique())

# mapping에 STATS_STATION_ID 부여
mapping_hc = station_mapping.merge(
    station_dict_valid,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="inner"
)

mapping_hc = (
    mapping_hc
    .sort_values(["CELL_ID", "nearest_station_distance_m", "STATS_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[HIGH CONFIDENCE MAPPING]")
print("mapping rows:", len(station_mapping))
print("hc rows:", len(mapping_hc))
print("unique stations:", mapping_hc["STATS_STATION_ID"].nunique())


# station activity aggregate
station_30_agg = (
    station_30
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] +
    station_30_agg["station_30_in_sum"]
)

station_daily_agg = (
    station_daily
    .dropna(subset=["STATION_ID"])
    .assign(STATS_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("STATS_STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] +
    station_daily_agg["station_daily_in_sum"]
)

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATS_STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)


# ============================================================
# 7. CELL mobility + station mapping 결합
# ============================================================

cell_station = mapping_hc.merge(
    cell_mobility,
    on="CELL_ID",
    how="left",
    suffixes=("", "_mobility")
)

for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_station[c] = pd.to_numeric(
        cell_station[c],
        errors="coerce"
    ).fillna(0)

print("\n[CELL-STATION JOIN]")
print("rows:", len(cell_station))
print("cells with mobility > 0:", (cell_station["actual_total_flow"] > 0).sum())
print("total mobility in matched cells:", cell_station["actual_total_flow"].sum())

cell_station.to_csv(
    cell_station_join_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8. STATION 단위 주변 mobility 집계
# ============================================================

station_mobility = (
    cell_station
    .groupby(["STATS_STATION_ID", "station_name", "dict_line"], as_index=False)
    .agg(
        nearby_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        surrounding_total_flow=("actual_total_flow", "sum"),
        surrounding_outflow=("mobility_out_sum", "sum"),
        surrounding_inflow=("mobility_in_sum", "sum"),
        surrounding_net_flow=("mobility_net_sum", "sum"),
        surrounding_abs_net_flow=("actual_abs_net_flow", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

print("\n[STATION MOBILITY]")
print(station_mobility.shape)
print(station_mobility.head())


# ============================================================
# 9. STATION mobility + subway activity 최종 결합
# ============================================================

station_level = station_mobility.merge(
    station_activity,
    on="STATS_STATION_ID",
    how="inner"
)

print("\n[STATION LEVEL]")
print(station_level.shape)


# ============================================================
# 10. 연구 지표 계산
# ============================================================

station_level["mobility_norm"] = normalize_minmax(
    station_level["surrounding_total_flow"]
)

station_level["subway_activity_norm"] = normalize_minmax(
    station_level["station_total_activity"]
)

station_level["distance_norm"] = normalize_minmax(
    station_level["mean_distance_m"]
)

station_level["subway_representation_gap"] = (
    station_level["mobility_norm"] -
    station_level["subway_activity_norm"]
)

station_level["hidden_demand_score"] = (
    station_level["mobility_norm"] *
    (1 - station_level["subway_activity_norm"])
)

station_level["distance_weighted_hidden_demand"] = (
    station_level["mobility_norm"] *
    station_level["distance_norm"]
)

station_level["subway_overrepresentation_score"] = (
    station_level["subway_activity_norm"] *
    (1 - station_level["mobility_norm"])
)

q_hidden = station_level["hidden_demand_score"].quantile(0.90)
q_gap = station_level["subway_representation_gap"].quantile(0.90)
q_over = station_level["subway_overrepresentation_score"].quantile(0.90)

conditions = [
    station_level["hidden_demand_score"] >= q_hidden,
    station_level["subway_representation_gap"] >= q_gap,
    station_level["subway_overrepresentation_score"] >= q_over
]

choices = [
    "hidden_demand_station",
    "subway_under_represented",
    "subway_over_represented"
]

station_level["station_type"] = np.select(
    conditions,
    choices,
    default="normal"
)

station_level = station_level.sort_values(
    "hidden_demand_score",
    ascending=False
)

top_hidden = station_level.head(100)


# ============================================================
# 11. 저장
# ============================================================

station_level.to_csv(
    station_result_path,
    index=False,
    encoding="utf-8-sig"
)

top_hidden.to_csv(
    top_hidden_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. 결과 확인
# ============================================================

print("\n================================================")
print("DONE")
print("================================================")
print("restored mobility:", restored_mobility_path)
print("cell-station join:", cell_station_join_path)
print("station result:", station_result_path)
print("top hidden:", top_hidden_path)

print("\n[FINAL SHAPE]")
print(station_level.shape)

print("\n[KEY METRICS]")
print(station_level[[
    "nearby_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "station_total_activity",
    "mobility_norm",
    "subway_activity_norm",
    "subway_representation_gap",
    "hidden_demand_score",
    "distance_weighted_hidden_demand",
    "subway_overrepresentation_score"
]].describe())

print("\n[STATION TYPE COUNTS]")
print(station_level["station_type"].value_counts())

print("\n[TOP HIDDEN DEMAND STATIONS]")
print(top_hidden[[
    "STATS_STATION_ID",
    "station_name",
    "dict_line",
    "nearby_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "station_total_activity",
    "mobility_norm",
    "subway_activity_norm",
    "subway_representation_gap",
    "hidden_demand_score",
    "station_type"
]].head(30))


[CELL MASTER]
(207205, 4)
   CELL_INDEX     CELL_ID  CELL_X   CELL_Y
0           0  나사99757775  899875  1977875
1           1  다사18001325  918125  1913375
2           2  다사31254225  931375  1942375
3           3  다사33752875  933875  1928875
4           4  다바40009800  940125  1898125

[MOBILITY RAW]
(630941, 5)
CELL_ID_BASE unique: 28486

[RESTORE CHECK]
original rows: 630941
numeric rows: 2067
string rows: 628874
restored rows: 630941
restored CELL_ID unique: 28475
missing CELL_ID: 0
missing CELL_X: 0

[CELL MOBILITY]
(28475, 9)
      CELL_ID  CELL_X   CELL_Y  mobility_out_sum  mobility_in_sum  \
0  가사47509825  747625  1998375               3.5              3.5   
1  가사47759700  747875  1997125               7.0              7.0   
2  가사47759875  747875  1998875               7.0              7.0   
3  가사48759800  748875  1998125              31.5             21.0   
4  가사50009525  750125  1995375              35.0             45.5   

   mobility_net_sum  record_count  actual_total_f

In [16]:
import os, glob
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로
# ============================================================

mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID.csv"

mobility_dir = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output"

subway_30min_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output"

subway_daily_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output"

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\all_period_compare_output"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# 1. 유틸
# ============================================================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            pass
    raise ValueError("CSV 읽기 실패: " + path)

def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def is_korean_cell_id(series):
    return series.astype(str).str.contains(r"[가-힣]", regex=True)

# ============================================================
# 2. mapping 로드
# ============================================================

mapping = read_csv_safe(mapping_path)

mapping["CELL_ID"] = mapping["CELL_ID"].astype(str).str.strip()
mapping["MASTER_STATION_ID"] = pd.to_numeric(mapping["MASTER_STATION_ID"], errors="coerce")

mapping = mapping.dropna(subset=["MASTER_STATION_ID"]).copy()
mapping["MASTER_STATION_ID"] = mapping["MASTER_STATION_ID"].astype(int)

mapping = (
    mapping
    .sort_values(["CELL_ID", "nearest_station_distance_m", "MASTER_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
)

print("[MAPPING]")
print(mapping.shape)
print("unique cells:", mapping["CELL_ID"].nunique())
print("unique stations:", mapping["MASTER_STATION_ID"].nunique())

# ============================================================
# 3. mobility 전체 parquet 읽기 → cell 단위 합산
# ============================================================

mobility_files = sorted(glob.glob(os.path.join(mobility_dir, "*.parquet")))
print("\n[MOBILITY FILES]", len(mobility_files))

mobility_parts = []

for i, path in enumerate(mobility_files, 1):
    try:
        df = pd.read_parquet(
            path,
            columns=["datetime", "CELL_ID_BASE", "out_cnt", "in_cnt", "net_cnt"]
        )

        df["CELL_ID_BASE_RAW"] = df["CELL_ID_BASE"].astype(str).str.strip()

        # 핵심: 한글 CELL_ID만 그대로 사용
        # 숫자형 CELL_ID_BASE는 전체에서 소수라 일단 제외
        mask = is_korean_cell_id(df["CELL_ID_BASE_RAW"])
        df = df[mask].copy()

        df["CELL_ID"] = df["CELL_ID_BASE_RAW"]

        for c in ["out_cnt", "in_cnt", "net_cnt"]:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

        agg = (
            df.groupby("CELL_ID", as_index=False)
            .agg(
                mobility_out_sum=("out_cnt", "sum"),
                mobility_in_sum=("in_cnt", "sum"),
                mobility_net_sum=("net_cnt", "sum"),
                mobility_record_count=("datetime", "count")
            )
        )

        agg["source_file"] = os.path.basename(path)
        mobility_parts.append(agg)

        if i % 50 == 0:
            print(f"loaded mobility {i}/{len(mobility_files)}")

    except Exception as e:
        print("skip mobility:", path)
        print("reason:", e)

mobility_all = pd.concat(mobility_parts, ignore_index=True)

cell_mobility = (
    mobility_all
    .groupby("CELL_ID", as_index=False)
    .agg(
        mobility_out_sum=("mobility_out_sum", "sum"),
        mobility_in_sum=("mobility_in_sum", "sum"),
        mobility_net_sum=("mobility_net_sum", "sum"),
        mobility_record_count=("mobility_record_count", "sum"),
        mobility_file_count=("source_file", "nunique")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)
cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

cell_mobility.to_csv(
    os.path.join(output_dir, "01_cell_mobility_all_period.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())

# ============================================================
# 4. cell → station mobility 집계
# ============================================================

cell_level = mapping.merge(cell_mobility, on="CELL_ID", how="left")

for c in [
    "mobility_out_sum", "mobility_in_sum", "mobility_net_sum",
    "mobility_record_count", "mobility_file_count",
    "actual_total_flow", "actual_abs_net_flow"
]:
    cell_level[c] = pd.to_numeric(cell_level[c], errors="coerce").fillna(0)

cell_level.to_csv(
    os.path.join(output_dir, "02_cell_level_with_station_all_period.csv"),
    index=False,
    encoding="utf-8-sig"
)

station_mobility = (
    cell_level
    .groupby(["MASTER_STATION_ID", "nearest_station", "nearest_line"], as_index=False)
    .agg(
        mapped_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        surrounding_total_flow=("actual_total_flow", "sum"),
        surrounding_outflow=("mobility_out_sum", "sum"),
        surrounding_inflow=("mobility_in_sum", "sum"),
        surrounding_netflow=("mobility_net_sum", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

station_mobility["active_cell_ratio"] = (
    station_mobility["active_cell_count"] / station_mobility["mapped_cell_count"]
)

station_mobility.to_csv(
    os.path.join(output_dir, "03_station_mobility_all_period.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("\n[STATION MOBILITY]")
print(station_mobility.shape)
print(station_mobility.head())

# ============================================================
# 5. subway 전체 parquet 읽기 함수
# ============================================================

def load_subway_folder(folder, label):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"\n[{label} FILES]", len(files))

    parts = []

    for i, path in enumerate(files, 1):
        try:
            df = pd.read_parquet(path)

            if "subwayid" in df.columns and "STATION_ID" not in df.columns:
                df = df.rename(columns={"subwayid": "STATION_ID"})

            needed = ["datetime", "STATION_ID", "od_out_cnt", "od_in_cnt", "od_net_cnt"]
            missing = [c for c in needed if c not in df.columns]
            if missing:
                print("skip:", os.path.basename(path), "missing:", missing)
                continue

            df = df[needed].copy()
            df["STATION_ID"] = pd.to_numeric(df["STATION_ID"], errors="coerce")

            for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
                df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

            agg = (
                df.dropna(subset=["STATION_ID"])
                .assign(MASTER_STATION_ID=lambda x: x["STATION_ID"].astype(int))
                .groupby("MASTER_STATION_ID", as_index=False)
                .agg(
                    subway_out_sum=("od_out_cnt", "sum"),
                    subway_in_sum=("od_in_cnt", "sum"),
                    subway_net_sum=("od_net_cnt", "sum"),
                    subway_record_count=("datetime", "count")
                )
            )

            agg["source_file"] = os.path.basename(path)
            parts.append(agg)

            if i % 50 == 0:
                print(f"loaded {label} {i}/{len(files)}")

        except Exception as e:
            print("skip subway:", path)
            print("reason:", e)

    if not parts:
        return pd.DataFrame()

    all_df = pd.concat(parts, ignore_index=True)

    out = (
        all_df.groupby("MASTER_STATION_ID", as_index=False)
        .agg(
            subway_out_sum=("subway_out_sum", "sum"),
            subway_in_sum=("subway_in_sum", "sum"),
            subway_net_sum=("subway_net_sum", "sum"),
            subway_record_count=("subway_record_count", "sum"),
            subway_file_count=("source_file", "nunique")
        )
    )

    out["subway_total_activity"] = out["subway_out_sum"] + out["subway_in_sum"]
    return out

# ============================================================
# 6. subway 30min / daily 전체 집계
# ============================================================

subway_30 = load_subway_folder(subway_30min_dir, "SUBWAY_30MIN")
subway_daily = load_subway_folder(subway_daily_dir, "SUBWAY_DAILY")

subway_30.to_csv(
    os.path.join(output_dir, "04_subway_30min_all_period.csv"),
    index=False,
    encoding="utf-8-sig"
)

subway_daily.to_csv(
    os.path.join(output_dir, "05_subway_daily_all_period.csv"),
    index=False,
    encoding="utf-8-sig"
)

# 두 지하철 데이터 합친 버전
subway_all = pd.concat(
    [
        subway_30.assign(source_type="30min"),
        subway_daily.assign(source_type="daily")
    ],
    ignore_index=True
)

subway_total = (
    subway_all.groupby("MASTER_STATION_ID", as_index=False)
    .agg(
        subway_out_sum=("subway_out_sum", "sum"),
        subway_in_sum=("subway_in_sum", "sum"),
        subway_net_sum=("subway_net_sum", "sum"),
        subway_record_count=("subway_record_count", "sum"),
        subway_file_count=("subway_file_count", "sum")
    )
)

subway_total["subway_total_activity"] = (
    subway_total["subway_out_sum"] + subway_total["subway_in_sum"]
)

subway_total.to_csv(
    os.path.join(output_dir, "06_subway_total_all_period.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("\n[SUBWAY TOTAL]")
print(subway_total.shape)
print(subway_total.head())

# ============================================================
# 7. 최종 비교 함수
# ============================================================

def make_compare(station_mobility, subway_df, label):
    final = station_mobility.merge(
        subway_df,
        on="MASTER_STATION_ID",
        how="inner"
    )

    final["mobility_norm"] = normalize_minmax(final["surrounding_total_flow"])
    final["subway_norm"] = normalize_minmax(final["subway_total_activity"])

    final["mobility_minus_subway_gap"] = final["mobility_norm"] - final["subway_norm"]

    final["hidden_demand_score"] = (
        final["mobility_norm"] * (1 - final["subway_norm"])
    )

    final["subway_overrepresentation_score"] = (
        final["subway_norm"] * (1 - final["mobility_norm"])
    )

    final["distance_norm"] = normalize_minmax(final["mean_distance_m"])

    final["distance_weighted_hidden_demand"] = (
        final["mobility_norm"] * final["distance_norm"]
    )

    q_hidden = final["hidden_demand_score"].quantile(0.90) if len(final) else 0
    q_over = final["subway_overrepresentation_score"].quantile(0.90) if len(final) else 0

    final["station_type"] = "normal"
    final.loc[final["hidden_demand_score"] >= q_hidden, "station_type"] = "mobility_high_subway_low"
    final.loc[final["subway_overrepresentation_score"] >= q_over, "station_type"] = "subway_high_mobility_low"

    final = final.sort_values("hidden_demand_score", ascending=False)

    out_path = os.path.join(output_dir, f"07_compare_{label}.csv")
    final.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\n[COMPARE {label}]")
    print(final.shape)
    print(final["station_type"].value_counts())
    print(final[[
        "MASTER_STATION_ID", "nearest_station", "nearest_line",
        "surrounding_total_flow", "subway_total_activity",
        "mobility_norm", "subway_norm",
        "mobility_minus_subway_gap", "hidden_demand_score",
        "station_type"
    ]].head(20))

    return final

# ============================================================
# 8. 비교 실행
# ============================================================

compare_30 = make_compare(station_mobility, subway_30, "subway_30min")
compare_daily = make_compare(station_mobility, subway_daily, "subway_daily")
compare_total = make_compare(station_mobility, subway_total, "subway_total")

# ============================================================
# 9. 겹쳐서 반복 등장하는 hidden station 확인
# ============================================================

hidden_30 = set(compare_30.loc[compare_30["station_type"] == "mobility_high_subway_low", "MASTER_STATION_ID"])
hidden_daily = set(compare_daily.loc[compare_daily["station_type"] == "mobility_high_subway_low", "MASTER_STATION_ID"])
hidden_total = set(compare_total.loc[compare_total["station_type"] == "mobility_high_subway_low", "MASTER_STATION_ID"])

robust_hidden = hidden_30 & hidden_daily & hidden_total

robust_hidden_df = compare_total[
    compare_total["MASTER_STATION_ID"].isin(robust_hidden)
].copy()

robust_hidden_df.to_csv(
    os.path.join(output_dir, "08_robust_hidden_demand_stations.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("\n[ROBUST HIDDEN STATIONS]")
print("count:", len(robust_hidden_df))
print(robust_hidden_df[[
    "MASTER_STATION_ID", "nearest_station", "nearest_line",
    "surrounding_total_flow", "subway_total_activity",
    "hidden_demand_score"
]].head(50))

print("\nDONE")
print("saved:", output_dir)

[MAPPING]
(85543, 21)
unique cells: 85543
unique stations: 766

[MOBILITY FILES] 1374
loaded mobility 50/1374
loaded mobility 100/1374
loaded mobility 150/1374
loaded mobility 200/1374
loaded mobility 250/1374
loaded mobility 300/1374
loaded mobility 350/1374
skip mobility: C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202307__202307__PURPOSE_forn_250M_20230718_cell.parquet
reason: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
loaded mobility 400/1374
loaded mobility 450/1374
loaded mobility 500/1374
loaded mobility 550/1374
loaded mobility 600/1374
loaded mobility 650/1374
loaded mobility 700/1374
loaded mobility 750/1374
loaded mobility 800/1374
loaded mobility 850/1374
loaded mobility 900/1374
loaded mobility 950/1374
loaded mobility 1000/1374
loaded mobility 1050/1374
loaded mobility 1100/1374
loaded mobility 1150/1374
loaded mobility 1200/1374
loaded mobility 1250/1374
loa

In [20]:
import os
import glob
import pandas as pd
from huggingface_hub import HfApi, create_repo

# =========================================================
# 0. 설정
# =========================================================

INPUT_DIR = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output"
OUTPUT_DIR = r"C:\Users\82108\Desktop\hf_ready_subway_stats"

REPO_ID = "youngbongbong/TBDM_TRANSIT_STAT_SUBWAY"
REPO_TYPE = "dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 1. 하루 단위 parquet → 월별 parquet로 압축 저장
# =========================================================

files = sorted(glob.glob(
    os.path.join(INPUT_DIR, "TBDM_TRANSIT_STAT_SUBWAY_*_station_stats.parquet")
))

print("total files:", len(files))

USE_COLUMNS = ["datetime", "STATION_ID", "od_out_cnt", "od_in_cnt", "od_net_cnt"]

month_groups = {}

for f in files:
    name = os.path.basename(f)
    date_str = name.split("_")[4][:6]
    month_groups.setdefault(date_str, []).append(f)

print("months:", len(month_groups))

for month, month_files in month_groups.items():
    dfs = []

    for f in month_files:
        try:
            df = pd.read_parquet(f, columns=USE_COLUMNS)
            dfs.append(df)
        except Exception as e:
            print("skip:", f, e)

    if not dfs:
        continue

    merged = pd.concat(dfs, ignore_index=True)

    save_path = os.path.join(OUTPUT_DIR, f"subway_stats_{month}.parquet")
    merged.to_parquet(save_path, compression="zstd", index=False)

    print("saved:", save_path, merged.shape)

# =========================================================
# 2. README.md 생성
# =========================================================

readme = """---
license: other
task_categories:
- tabular-classification
- time-series-forecasting
- feature-extraction
tags:
- urban-mobility
- transportation
- subway
- seoul
- spatio-temporal
- geospatial
- hidden-demand
pretty_name: Seoul Subway Station Statistics
---

# Seoul Subway Station Statistics

This dataset contains processed subway station-level boarding and alighting statistics for urban mobility research.

## Data Fields

- `datetime`: timestamp
- `STATION_ID`: subway station identifier
- `od_out_cnt`: outgoing passenger count
- `od_in_cnt`: incoming passenger count
- `od_net_cnt`: net passenger flow

## Intended Use

This dataset is intended for non-commercial research and educational purposes, including:

- urban mobility analysis
- subway demand analysis
- hidden demand estimation
- spatio-temporal transportation modeling
- public transit accessibility analysis

## License and Usage

Users must comply with the original data provider's terms of use.  
This repository provides processed files for research-oriented analysis only.
"""

with open(os.path.join(OUTPUT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)

# =========================================================
# 3. Hugging Face 업로드
# =========================================================

api = HfApi()

create_repo(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    exist_ok=True
)

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    path_in_repo="subway_stats",
    commit_message="Upload monthly subway station statistics parquet files"
)

print("DONE")
print("uploaded to:", f"https://huggingface.co/datasets/{REPO_ID}")

total files: 3146
months: 104
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201701.parquet (380305, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201702.parquet (342847, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201703.parquet (379488, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201704.parquet (366796, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201705.parquet (378790, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201706.parquet (367461, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201707.parquet (379085, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201708.parquet (379512, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201709.parquet (382126, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stats\subway_stats_201710.parquet (391809, 5)
saved: C:\Users\82108\Desktop\hf_ready_subway_stat

HfHubHTTPError: (Request ID: Root=1-69fbd142-585e4c593224e4d9319d1356;0bd19af5-1465-4e7e-82ae-7be90f843c0a)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/datasets/youngbongbong/TBDM_TRANSIT_STAT_SUBWAY/preupload/main.
Make sure your token has the correct permissions.

In [21]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 0. 경로 설정
# =========================

input_path = r"C:\Users\82108\Desktop\새 폴더 (8)\all_period_compare_output\COMPARE_subway_total.csv"

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\visualization_output"
os.makedirs(output_dir, exist_ok=True)

# =========================
# 1. 파일 읽기
# =========================

if input_path.endswith(".xlsx"):
    df = pd.read_excel(input_path)
else:
    df = pd.read_csv(input_path, encoding="utf-8-sig")

print(df.shape)
print(df.columns.tolist())
print(df.head())

# =========================
# 2. 숫자형 정리
# =========================

num_cols = [
    "surrounding_total_flow",
    "subway_total_activity",
    "mobility_norm",
    "subway_norm",
    "mobility_minus_subway_gap",
    "hidden_demand_score",
    "subway_overrepresentation_score",
    "mean_distance_m"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# =========================
# 3. Scatter: 생활이동 vs 지하철 이용량
# =========================

plt.figure(figsize=(8, 6))
plt.scatter(
    df["subway_total_activity"],
    df["surrounding_total_flow"],
    alpha=0.6
)
plt.xlabel("Subway total activity")
plt.ylabel("Surrounding mobility flow")
plt.title("Mobility vs Subway Activity")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "01_mobility_vs_subway_scatter.png"), dpi=300)
plt.show()

# =========================
# 4. Top hidden demand stations
# =========================

top_hidden = df.sort_values("hidden_demand_score", ascending=False).head(20)

plt.figure(figsize=(10, 7))
plt.barh(
    top_hidden["nearest_station"].astype(str)[::-1],
    top_hidden["hidden_demand_score"][::-1]
)
plt.xlabel("Hidden demand score")
plt.ylabel("Station")
plt.title("Top Hidden Demand Stations")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "02_top_hidden_demand_stations.png"), dpi=300)
plt.show()

# =========================
# 5. Top subway overrepresented stations
# =========================

top_over = df.sort_values("subway_overrepresentation_score", ascending=False).head(20)

plt.figure(figsize=(10, 7))
plt.barh(
    top_over["nearest_station"].astype(str)[::-1],
    top_over["subway_overrepresentation_score"][::-1]
)
plt.xlabel("Subway overrepresentation score")
plt.ylabel("Station")
plt.title("Top Subway Overrepresented Stations")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "03_top_subway_overrepresented_stations.png"), dpi=300)
plt.show()

# =========================
# 6. Station type count
# =========================

type_counts = df["station_type"].value_counts()

plt.figure(figsize=(7, 5))
plt.bar(type_counts.index.astype(str), type_counts.values)
plt.xlabel("Station type")
plt.ylabel("Count")
plt.title("Station Type Distribution")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "04_station_type_distribution.png"), dpi=300)
plt.show()

# =========================
# 7. Gap ranking
# =========================

top_gap = df.sort_values("mobility_minus_subway_gap", ascending=False).head(20)

plt.figure(figsize=(10, 7))
plt.barh(
    top_gap["nearest_station"].astype(str)[::-1],
    top_gap["mobility_minus_subway_gap"][::-1]
)
plt.xlabel("Mobility - Subway Gap")
plt.ylabel("Station")
plt.title("Top Mobility-Subway Gap Stations")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "05_top_mobility_subway_gap.png"), dpi=300)
plt.show()

print("DONE")
print("saved to:", output_dir)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\82108\\Desktop\\새 폴더 (8)\\all_period_compare_output\\COMPARE_subway_total.csv'